# JobSpy Interactive Job Scraper

Interactive job scraper using [JobSpy](https://github.com/speedyapply/JobSpy) with interactive widgets.
Scrape jobs from LinkedIn, Indeed, Glassdoor, Google, ZipRecruiter & more.

**Usage:** Adjust the parameters below using the interactive widgets and run the scraper cell.

In [ ]:
# Install dependencies
!pip install -U python-jobspy ipywidgets -q

In [ ]:
import csv
import pandas as pd
from jobspy import scrape_jobs
from datetime import datetime
from IPython.display import display, HTML, JSON
import ipywidgets as widgets
from ipywidgets import interact, interact_manual, HBox, VBox, Label

## 🔍 Interactive Search Parameters

Use the widgets below to configure your search:

In [ ]:
# Create interactive widgets
search_term_widget = widgets.Text(
    value='data analyst',
    placeholder='e.g., data analyst, software engineer',
    description='Search Term:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%')
)

location_widget = widgets.Text(
    value='Remote',
    placeholder='e.g., Remote, San Francisco CA, India',
    description='Location:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%')
)

available_sites = [
    'indeed',
    'linkedin',
    'zip_recruiter',
    'google',
    'glassdoor',
    'bayt',
    'naukri',
    'bdjobs'
]

sites_widget = widgets.SelectMultiple(
    options=available_sites,
    value=['indeed', 'linkedin', 'zip_recruiter', 'google'],
    description='Job Sites:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', height='150px')
)

results_widget = widgets.IntSlider(
    value=50,
    min=10,
    max=100,
    step=10,
    description='Results per site:',
    style={'description_width': 'initial'}
)

hours_widget = widgets.IntSlider(
    value=72,
    min=24,
    max=168,
    step=24,
    description='Hours old:',
    style={'description_width': 'initial'}
)

country_widget = widgets.Dropdown(
    options=['USA', 'India', 'UK', 'Canada', 'Australia', 'Germany', 'France', 'Netherlands', 'Singapore', 'Brazil', 'Mexico'],
    value='USA',
    description='Country:',
    style={'description_width': 'initial'}
)

google_search_widget = widgets.Text(
    value='',
    placeholder='e.g., software engineer jobs near San Francisco, CA since yesterday',
    description='Google Search:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%')
)

remote_widget = widgets.Checkbox(
    value=True,
    description='Remote jobs only',
    style={'description_width': 'initial'}
)

linkedin_desc_widget = widgets.Checkbox(
    value=False,
    description='LinkedIn: Fetch full descriptions (slower)',
    style={'description_width': 'initial'}
)

proxies_widget = widgets.Text(
    value='',
    placeholder='e.g., user:pass@host:port, localhost',
    description='Proxies:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%')
)

# Display widgets
display(VBox([
    Label('🔍 Configure Your Search'),
    search_term_widget,
    location_widget,
    sites_widget,
    HBox([results_widget, hours_widget]),
    country_widget,
    google_search_widget,
    HBox([remote_widget, linkedin_desc_widget]),
    proxies_widget
]))

## 🚀 Run Scraper

Click the button below to scrape jobs with your configured parameters:

In [ ]:
# Scrape button
scrape_button = widgets.Button(
    description='🔍 Scrape Jobs',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

output = widgets.Output()

def on_scrape_click(b):
    with output:
        output.clear_output()
        print(f"🔍 Searching for '{search_term_widget.value}' jobs in '{location_widget.value}'...")
        print(f"📊 Sites: {', '.join(sites_widget.value) if sites_widget.value else 'None selected'}")
        print(f"⏰ Jobs posted in last {hours_widget.value} hours")
        print("\n⏳ This may take 1-3 minutes...\n")
        
        # Parse proxies
        proxies = None
        if proxies_widget.value and proxies_widget.value.strip():
            proxies = [p.strip() for p in proxies_widget.value.split(',') if p.strip()]
        
        # Prepare parameters
        scrape_params = {
            'site_name': list(sites_widget.value) if sites_widget.value else ['indeed'],
            'search_term': search_term_widget.value or 'data analyst',
            'location': location_widget.value or 'Remote',
            'results_wanted': results_widget.value,
            'hours_old': hours_widget.value,
            'country_indeed': country_widget.value,
            'is_remote': remote_widget.value,
            'linkedin_fetch_description': linkedin_desc_widget.value,
            'verbose': 1
        }
        
        if google_search_widget.value and google_search_widget.value.strip():
            scrape_params['google_search_term'] = google_search_widget.value
        
        if proxies:
            scrape_params['proxies'] = proxies
        
        try:
            jobs = scrape_jobs(**scrape_params)
            
            if len(jobs) > 0:
                print(f"\n✅ Found {len(jobs)} jobs!")
                
                # Display summary
                print(f"\n📈 Summary:")
                print(f"   Total jobs: {len(jobs)}")
                
                if 'site' in jobs.columns:
                    print(f"\n📊 Jobs by source:")
                    print(jobs['site'].value_counts().to_string())
                
                if 'job_type' in jobs.columns:
                    print(f"\n💼 Job types:")
                    print(jobs['job_type'].value_counts().to_string())
                
                # Store jobs in global variable
                globals()['scraped_jobs'] = jobs
                
                # Display table
                display_cols = ['title', 'company', 'location', 'site', 'job_type', 'job_url']
                if 'min_amount' in jobs.columns and 'max_amount' in jobs.columns:
                    display_cols.extend(['min_amount', 'max_amount'])
                if 'date_posted' in jobs.columns:
                    display_cols.append('date_posted')
                
                display_cols = [col for col in display_cols if col in jobs.columns]
                display(jobs[display_cols].head(100))
                
                # Export options
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                search_safe = search_term_widget.value.replace(' ', '_') if search_term_widget.value else 'jobs'
                csv_filename = f"jobs_{search_safe}_{timestamp}.csv"
                
                jobs.to_csv(csv_filename, quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
                print(f"\n💾 Exported to: {csv_filename}")
                
            else:
                print("\n⚠️ No jobs found. Try adjusting your search parameters.")
                globals()['scraped_jobs'] = pd.DataFrame()
                
        except Exception as e:
            print(f"\n❌ Error: {e}")
            import traceback
            traceback.print_exc()
            globals()['scraped_jobs'] = pd.DataFrame()

scrape_button.on_click(on_scrape_click)

display(VBox([scrape_button, output]))

## 📊 View Results

After scraping, view the full results table:

In [ ]:
# Display scraped jobs if available
if 'scraped_jobs' in globals() and len(scraped_jobs) > 0:
    display_cols = ['title', 'company', 'location', 'site', 'job_type', 'job_url']
    if 'min_amount' in scraped_jobs.columns and 'max_amount' in scraped_jobs.columns:
        display_cols.extend(['min_amount', 'max_amount'])
    if 'date_posted' in scraped_jobs.columns:
        display_cols.append('date_posted')
    
    display_cols = [col for col in display_cols if col in scraped_jobs.columns]
    display(scraped_jobs[display_cols])
else:
    print("No jobs scraped yet. Use the scraper above to fetch jobs.")

## 💾 Export Results

Export your scraped jobs:

In [ ]:
if 'scraped_jobs' in globals() and len(scraped_jobs) > 0:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    search_safe = search_term_widget.value.replace(' ', '_') if search_term_widget.value else 'jobs'
    
    csv_filename = f"jobs_{search_safe}_{timestamp}.csv"
    json_filename = f"jobs_{search_safe}_{timestamp}.json"
    
    scraped_jobs.to_csv(csv_filename, quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)
    scraped_jobs.to_json(json_filename, orient='records', indent=2, date_format='iso', default=str)
    
    print(f"✅ Exported to:")
    print(f"   📄 {csv_filename}")
    print(f"   📄 {json_filename}")
    
    display(HTML(f"<p><a href='{csv_filename}' download>📥 Download CSV</a> | <a href='{json_filename}' download>📥 Download JSON</a></p>"))
else:
    print("No jobs to export. Scrape jobs first using the button above.")